In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.5.0
pandas: 3.0.3


In [2]:
# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


In [3]:
# 시나리오 1 — 검증 → 병합
# (1) 검증: 룩업 표의 키가 유일한가, 주문의 키가 모두 매칭되는가
print("[병합 전 검증]")
print("  customers 키 중복:", cust["customer_id"].duplicated().sum(), "건")
print("  products  키 중복:", prod["product_id"].duplicated().sum(), "건")
print("  매칭 안 되는 customer_id:", (~ordr["customer_id"].isin(cust["customer_id"])).sum(), "건")
print("  매칭 안 되는 product_id :", (~ordr["product_id"].isin(prod["product_id"])).sum(), "건")

# (2) 병합: validate로 관계 가정(m:1)을 강제. indicator로 매칭 확인.
df = (
    ordr
    .merge(cust, on="customer_id", how="left", validate="m:1")
    .merge(prod, on="product_id", how="left", validate="m:1")
)
print("\n[병합 결과] 행 수:", len(df), "(원본 주문 수와 같으면 폭증 없음)")
display(df.head(3))

[병합 전 검증]
  customers 키 중복: 0 건
  products  키 중복: 0 건
  매칭 안 되는 customer_id: 0 건
  매칭 안 되는 product_id : 0 건

[병합 결과] 행 수: 1500 (원본 주문 수와 같으면 폭증 없음)


,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,패션,40000
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,식품,25000
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,패션,12000


In [4]:
# 시나리오 2 — 월별 KPI
df["month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_kpi = (
    df.groupby("month")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),
        구매고객수=("customer_id", "nunique"),   # nunique: 고유 고객 수
    )
    .round(0)
)
# 전월 대비 매출 증감률(%)도 추가 — 경영 보고서의 단골 지표
monthly_kpi["매출증감률(%)"] = (monthly_kpi["총매출"].pct_change() * 100).round(1)

print("[월별 KPI 요약표]")
display(monthly_kpi)

[월별 KPI 요약표]


,총매출,주문건수,객단가,구매고객수,매출증감률(%)
month,,,,,
2025-01,19902000.0,330,60309.0,163,NaN
2025-02,17043000.0,289,58972.0,160,-14.4
2025-03,20035000.0,326,61457.0,156,17.6
2025-04,17602000.0,287,61331.0,156,-12.1
2025-05,16374000.0,268,61097.0,146,-7.0


In [5]:
# 시나리오 3-1 — 월 × 카테고리 매출 교차표 (pivot_table)
month_category = pd.pivot_table(
    df, index="month", columns="category", values="amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="합계",
).round(0)
print("[월 × 카테고리 매출 교차표]")
display(month_category)

# 시나리오 3-2 — 각 주문이 '그 달 전체 매출'에서 차지하는 비중 (transform)
df["month_total"] = df.groupby("month")["amount"].transform("sum")
df["share_in_month(%)"] = (df["amount"] / df["month_total"] * 100).round(3)
print("\n[주문별 월 매출 점유율 — 앞 5행]")
display(df[["order_id", "month", "amount", "share_in_month(%)"]].head())

[월 × 카테고리 매출 교차표]


category,가전,뷰티,식품,패션,합계
month,,,,,
2025-01,5001000.0,2183000.0,4436000.0,8282000.0,19902000.0
2025-02,3752000.0,2433000.0,4778000.0,6080000.0,17043000.0
2025-03,4572000.0,1482000.0,4531000.0,9450000.0,20035000.0
2025-04,3998000.0,1954000.0,4858000.0,6792000.0,17602000.0
2025-05,2925000.0,1934000.0,3801000.0,7714000.0,16374000.0
합계,20248000.0,9986000.0,22404000.0,38318000.0,90956000.0



[주문별 월 매출 점유율 — 앞 5행]


,order_id,month,amount,share_in_month(%)
0,O00001,2025-04,120000.0,0.682
1,O00002,2025-03,25000.0,0.125
2,O00003,2025-01,36000.0,0.181
3,O00004,2025-03,120000.0,0.599
4,O00005,2025-03,120000.0,0.599


In [6]:
# 핵심 숫자 자동 추출 (보고서 문장에 채워 넣기 위함)
display(monthly_kpi)
display(df)

best_month = monthly_kpi["총매출"].idxmax()
best_month_sales = monthly_kpi["총매출"].max()
top_category = df.groupby("category")["amount"].sum().idxmax()
top_region = df.groupby("region")["amount"].sum().idxmax()
overall_aov = df["amount"].mean()

print("자동 추출된 핵심 숫자")
print(f"  최대 매출 월   : {best_month} ({best_month_sales:,.0f}원)")
print(f"  최대 매출 카테고리: {top_category}")
print(f"  최대 매출 지역 : {top_region}")
print(f"  전체 객단가    : {overall_aov:,.0f}원")

,총매출,주문건수,객단가,구매고객수,매출증감률(%)
month,,,,,
2025-01,19902000.0,330,60309.0,163,NaN
2025-02,17043000.0,289,58972.0,160,-14.4
2025-03,20035000.0,326,61457.0,156,17.6
2025-04,17602000.0,287,61331.0,156,-12.1
2025-05,16374000.0,268,61097.0,146,-7.0


,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,category,price,month,month_total,share_in_month(%)
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,패션,40000,2025-04,17602000.0,0.682
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,식품,25000,2025-03,20035000.0,0.125
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,패션,12000,2025-01,19902000.0,0.181
3,O00004,C0123,P002,3,120000.0,2025-03-26,경기,basic,식품,40000,2025-03,20035000.0,0.599
4,O00005,C0150,P014,3,120000.0,2025-03-16,경기,basic,가전,40000,2025-03,20035000.0,0.599
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,O01496,C0082,P013,3,120000.0,2025-05-17,경기,basic,가전,40000,2025-05,16374000.0,0.733
1496,O01497,C0067,P011,3,120000.0,2025-05-26,인천,premium,패션,40000,2025-05,16374000.0,0.733
1497,O01498,C0065,P013,1,40000.0,2025-01-14,서울,premium,가전,40000,2025-01,19902000.0,0.201
1498,O01499,C0006,P014,1,40000.0,2025-03-01,인천,premium,가전,40000,2025-03,20035000.0,0.200


자동 추출된 핵심 숫자
  최대 매출 월   : 2025-03 (20,035,000원)
  최대 매출 카테고리: 패션
  최대 매출 지역 : 인천
  전체 객단가    : 60,637원


# 데이터 분석 보고서

# 1. 분석 개요
- 대상: 모두마켓 주문 1,500건 (2025-01 ~ 2025-05), 고객·상품 정보 병합
- 방법: 3개 테이블 키 검증 후 m:1 병합 → 월별 KPI 집계 → 교차표·점유율 분석

# 2. 핵심 발견
- 매출이 가장 높았던 달은 "25년 3월"이었으며, 전월 대비 증감률은 17.6% 였음.
- 카테고리 중 "패션", 지역 중 "인천" 의 매출 기여가 가장 컸음.
- 전체 객단가는 약 60,637원.

# 3. 데이터 신뢰성
- 병합 전 키 검증: 중복 0건, 미매칭 0건 → 행 폭증·조용한 결측 없음.
- validate="m:1" 통과 → 관계 가정(주문:상품 = m:1) 유효.
- 병합 전/후 행수 동일함

# 4. 제언
- 월별 매출 변동의 원인(고객 수 ↑ vs 객단가 ↑)을 구매고객수·객단가 열로 분해해 마케팅 전략 차별화 필요함